In [1]:
import pandas as pd
import csv
import pandas_gbq
import time
from datetime import datetime
import os
from google.oauth2 import service_account
from google.cloud import bigquery
from datetime import datetime
from dateutil.relativedelta import relativedelta
import numpy as np

In [2]:
CREDS = '../../converge-database-0331482f2ee5.json'
client = bigquery.Client.from_service_account_json(json_credentials_path=CREDS)

In [3]:
combined = pd.DataFrame()
set_month = "202601"

In [30]:
combined = pd.read_excel('I:New Structure/Actuarial New/Database/Farmers/'+set_month+'/01 26 Converge FIA Revised.xlsx', sheet_name='Combined GL')

In [31]:
combined.columns = combined.columns.map(str.lower)
combined.columns = combined.columns.map(lambda x : x.replace(" " , "_"))

In [32]:
combined

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,...,converge_debit,converge_credit,converge_net,policy_plan,code,issue_date,ceding_allowance,unnamed:_20,unnamed:_21,unnamed:_22
0,7368,Purchasing,2026-01-05,1-C00-5100-102,Surrender Benefits Paid-FIA,1991.83,0.00,1991.83,Annuity Partial Surrender,Annuity Partial Surrender,...,1792.647,0.000,1792.647,AV7Q0R,HFIA07.0,2024-11-21,0.00000,NaN,HFIA05.0,0.0275
1,7416,Purchasing,2026-01-07,1-C00-5100-102,Surrender Benefits Paid-FIA,5178.24,0.00,5178.24,Annuity Partial Surrender,Annuity Partial Surrender,...,4660.416,0.000,4660.416,AF1Q1R,HFIA10.2,2023-12-07,0.00000,NaN,HFIA05.1,0.0275
2,7417,Purchasing,2026-01-07,1-C00-5100-102,Surrender Benefits Paid-FIA,2414.83,0.00,2414.83,Annuity Partial Surrender,Annuity Partial Surrender,...,2173.347,0.000,2173.347,AF1N2R,HFIA10.2,2024-03-21,0.00000,NaN,HFIA07.0,0.0275
3,7466,Purchasing,2026-01-09,1-C00-5100-102,Surrender Benefits Paid-FIA,693.76,0.00,693.76,Annuity RMD,Annuity RMD,...,624.384,0.000,624.384,AV1Q2R,HFIA10.2,2024-06-28,0.00000,NaN,HFIA07.1,0.0275
4,7489,Purchasing,2026-01-13,1-C00-5100-102,Surrender Benefits Paid-FIA,1160.37,0.00,1160.37,Annuity Partial Surrender,Annuity Partial Surrender,...,1044.333,0.000,1044.333,AV1Q1R,HFIA10.2,2024-09-28,0.00000,NaN,HFIA07.2,0.0275
5,7492,Purchasing,2026-01-13,1-C00-5100-102,Surrender Benefits Paid-FIA,15000.00,0.00,15000.00,Annuity Partial Surrender,Annuity Partial Surrender,...,13500.000,0.000,13500.000,AV1Q1R,HFIA10.2,2025-01-07,0.00000,NaN,HFIA10.0,0.0275
6,7530,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,4125.85,0.00,4125.85,Annuity Partial Surrender,Annuity Partial Surrender,...,3713.265,0.000,3713.265,AF1Q1R,HFIA10.2,2023-12-07,0.00000,NaN,HFIA10.1,0.0275
7,7531,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,10400.00,0.00,10400.00,Annuity Partial Surrender,Annuity Partial Surrender,...,9360.000,0.000,9360.000,AF1N1R,HFIA10.2,2023-12-07,0.00000,NaN,HFIA10.2,0.0275
8,7533,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,3725.86,0.00,3725.86,Annuity Partial Surrender,Annuity Partial Surrender,...,3353.274,0.000,3353.274,AV1Q2R,HFIA10.2,2024-07-28,0.00000,NaN,NaN,NaN
9,7534,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,4041.33,0.00,4041.33,Annuity Partial Surrender,Annuity Partial Surrender,...,3637.197,0.000,3637.197,AV1Q1R,HFIA10.2,2024-11-21,0.00000,NaN,NaN,NaN


In [33]:
combined.debit_amount.replace(" -   ", np.nan, inplace=True)
combined.credit_amount.replace(" -   ", np.nan, inplace=True)

In [34]:
combined = combined.astype({"trx_date" : "datetime64[ns]", "debit_amount" : "float64", "credit_amount" : "float64" ,"originating_master_id" :"str", "converge_quota_share" : "float64","net" : "float64"})


In [35]:
combined = combined.iloc[:, 0:19]

In [36]:
combined = combined.iloc[:, 0:19]
combined['set_month'] = set_month

In [37]:
distribution_types = {
    'Annuity Full Surrender': 'full_surrender',
    'Annuity Partial Surrender': 'partial_surrender',
    'Annuity RMD': 'rmd',
    'Annuity Regular Distribution': 'other'
}

In [38]:
for desc, table_name in distribution_types.items():

    df = combined[
        (combined['account_number'] == '1-C00-5100-102') &
        (combined['description'] == desc)
    ].copy()

    df.rename(columns={
        "policy_id": "policy_number",
        "converge_quota_share": "quota_share",
        "net": "av_withdrawn",
        "policy_plan": "plan"
    }, inplace=True)

    df = df[[
        "policy_number",
        "plan",
        "av_withdrawn",
        "quota_share"
    ]]
    df["set_month"] = set_month
    df.info()
    try:
        df.to_gbq(
            destination_table=f"farmers_fia.{table_name}",
            project_id="converge-database",
            if_exists="append"
        )

        print(f"✅ Uploaded {desc} ({len(df)} rows) -> farmers_fia.{table_name}")

    except Exception as e:
        print(f"❌ Failed to upload {desc}: {e}")

<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  0 non-null      object 
 1   plan           0 non-null      object 
 2   av_withdrawn   0 non-null      float64
 3   quota_share    0 non-null      float64
 4   set_month      0 non-null      object 
dtypes: float64(2), object(3)
memory usage: 0.0+ bytes
✅ Uploaded Annuity Full Surrender (0 rows) -> farmers_fia.full_surrender
<class 'pandas.core.frame.DataFrame'>
Int64Index: 19 entries, 0 to 25
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  19 non-null     object 
 1   plan           19 non-null     object 
 2   av_withdrawn   19 non-null     float64
 3   quota_share    19 non-null     float64
 4   set_month      19 non-null     object 
dtypes: float64(2), object(3)
memory usage: 912.0+ bytes


100%|██████████| 1/1 [00:00<?, ?it/s]


✅ Uploaded Annuity Partial Surrender (19 rows) -> farmers_fia.partial_surrender
<class 'pandas.core.frame.DataFrame'>
Int64Index: 2 entries, 3 to 13
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  2 non-null      object 
 1   plan           2 non-null      object 
 2   av_withdrawn   2 non-null      float64
 3   quota_share    2 non-null      float64
 4   set_month      2 non-null      object 
dtypes: float64(2), object(3)
memory usage: 96.0+ bytes


100%|██████████| 1/1 [00:00<?, ?it/s]


✅ Uploaded Annuity RMD (2 rows) -> farmers_fia.rmd
<class 'pandas.core.frame.DataFrame'>
Int64Index: 0 entries
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   policy_number  0 non-null      object 
 1   plan           0 non-null      object 
 2   av_withdrawn   0 non-null      float64
 3   quota_share    0 non-null      float64
 4   set_month      0 non-null      object 
dtypes: float64(2), object(3)
memory usage: 0.0+ bytes
✅ Uploaded Annuity Regular Distribution (0 rows) -> farmers_fia.other


In [39]:
premium = combined[
    (combined['account_number'].isin([
        '1-C00-4000-102',
        '1-C00-4001-102'
    ])) &
    (combined['converge_quota_share'] > 0)
]

In [40]:
premium

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,originating_master_id,policy_id,converge_quota_share,converge_debit,converge_credit,converge_net,policy_plan,code,issue_date,set_month
32,7733,Financial,2026-01-06,1-C00-4001-102,Premium Income,0.0,10716.80,-10716.80,Annuity Prems-FIA-Qual,19885F,1062601,19885F,0.9,0.0,9645.120,-9645.120,AV1Q1R,HFIA10.2,2025-02-07,202601
47,7733,Financial,2026-01-19,1-C00-4001-102,Premium Income,0.0,11216.48,-11216.48,Annuity Prems-FIA-Qual,19885F,1192601,19885F,0.9,0.0,10094.832,-10094.832,AV1Q1R,HFIA10.2,2025-02-07,202601


In [41]:
premium = premium.rename(columns = {"net" : "gross_premium", "converge_quota_share" : "quota_share", "policy_id" : "policy_number"})
premium['set_month'] = set_month

In [42]:
premium = premium[['policy_number', 'gross_premium', 'quota_share', 'set_month']]

In [43]:
premium.to_gbq("converge-database.farmers.premium",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [44]:
death = combined[
        (combined['account_number'] == '1-C00-5000-102') &
        (combined['converge_quota_share'] > 0)
    ]

In [45]:
death

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,net,description,reference,originating_master_id,policy_id,converge_quota_share,converge_debit,converge_credit,converge_net,policy_plan,code,issue_date,set_month
16,7588,Purchasing,2026-01-20,1-C00-5000-102,Death Benefits: Paid-FIA,24680.11,0.0,24680.11,Death Claim,Death Claim,P13398F -001,13398F,0.9,22212.099,0.0,22212.099,AF1Q1R,HFIA10.2,2024-01-28,202601
17,7601,Purchasing,2026-01-20,1-C00-5000-102,Death Benefits: Paid-FIA,15764.03,0.0,15764.03,Death Claim,Death Claim,P18384F -001,18384F,0.9,14187.627,0.0,14187.627,AV1N2R,HFIA10.2,2024-11-07,202601


In [46]:
death = death.rename(columns = {"net" : "net_amount", "converge_quota_share" : "quotashare", "policy_id" : "policy_number", "issue_date" : "missuedt", "policy_plan" : "plan"})
death['set_month']=set_month

In [47]:
death = death[["policy_number", "net_amount", "missuedt", "plan", "quotashare", "set_month"]]

In [48]:
death.to_gbq("converge-database.farmers_fia.death_claims",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [23]:
combined = combined.iloc[:, 0:16]

In [24]:
combined['set_month'] = set_month
combined = combined.rename(columns = {"net" : "diff", "converge_quota_share" : "quota_share"})

In [25]:
combined = combined.drop(['converge_quota_share', 'converge_debit', 'converge_net', 'converge_credit'], axis=1, errors="ignore")

In [26]:
combined

,journal_entry,series,trx_date,account_number,account_description,debit_amount,credit_amount,diff,description,reference,originating_master_id,policy_id,quota_share,set_month
0,7368,Purchasing,2026-01-05,1-C00-5100-102,Surrender Benefits Paid-FIA,1991.83,0.00,1991.83,Annuity Partial Surrender,Annuity Partial Surrender,P17559F -001,17559F,0.9,202601
1,7416,Purchasing,2026-01-07,1-C00-5100-102,Surrender Benefits Paid-FIA,5178.24,0.00,5178.24,Annuity Partial Surrender,Annuity Partial Surrender,P13185F -002,13185F,0.9,202601
2,7417,Purchasing,2026-01-07,1-C00-5100-102,Surrender Benefits Paid-FIA,2414.83,0.00,2414.83,Annuity Partial Surrender,Annuity Partial Surrender,P13691F -001,13691F,0.9,202601
3,7466,Purchasing,2026-01-09,1-C00-5100-102,Surrender Benefits Paid-FIA,693.76,0.00,693.76,Annuity RMD,Annuity RMD,P15710F -001,15710F,0.9,202601
4,7489,Purchasing,2026-01-13,1-C00-5100-102,Surrender Benefits Paid-FIA,1160.37,0.00,1160.37,Annuity Partial Surrender,Annuity Partial Surrender,P16870F -001,16870F,0.9,202601
5,7492,Purchasing,2026-01-13,1-C00-5100-102,Surrender Benefits Paid-FIA,15000.00,0.00,15000.00,Annuity Partial Surrender,Annuity Partial Surrender,P19478F -001,19478F,0.9,202601
6,7530,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,4125.85,0.00,4125.85,Annuity Partial Surrender,Annuity Partial Surrender,P13094F -002,13094F,0.9,202601
7,7531,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,10400.00,0.00,10400.00,Annuity Partial Surrender,Annuity Partial Surrender,P13096F -002,13096F,0.9,202601
8,7533,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,3725.86,0.00,3725.86,Annuity Partial Surrender,Annuity Partial Surrender,P15845F -001,15845F,0.9,202601
9,7534,Purchasing,2026-01-14,1-C00-5100-102,Surrender Benefits Paid-FIA,4041.33,0.00,4041.33,Annuity Partial Surrender,Annuity Partial Surrender,P18731F -001,18731F,0.9,202601


In [27]:
combined.to_gbq("converge-database.farmers.combined_gl",
                  if_exists='append',
                  project_id="converge-database")

100%|██████████| 1/1 [00:00<?, ?it/s]


In [51]:
import sys

# Add directory containing LDTI.py
sys.path.append('../../actuarial-pipelines/reconciliations/farmers/')

# Import the function
from reconciliation import run_reconciliation

# Trigger the AVRF analysis
run_reconciliation(set_month, "farmers")


    SELECT SUM(net_amount) AS total
    FROM `farmers.death_claims`
    WHERE set_month = '202601'
    
{'product': 'farmers', 'fieldname': 'death_claims', 'total': 2290721.3000000003}
-----------------------------------------------

    SELECT SUM(net_amount) AS total
    FROM `farmers.cancellation`
    WHERE set_month = '202601'
    
{'product': 'farmers', 'fieldname': 'cancellation', 'total': None}
-----------------------------------------------

    select sum(av_withdrawn*quota_share) FROM `farmers.full_surrender` WHERE set_month = "202601"
    
{'product': 'farmers', 'fieldname': 'full_surrender', 'total': 199005.82650000002}
-----------------------------------------------

    select sum(gross_premium*quota_share) FROM `farmers.premium` WHERE set_month = "202601"
    
{'product': 'farmers', 'fieldname': 'premium', 'total': -49602.308}
-----------------------------------------------

    select sum(av_withdrawn*quota_share) FROM `farmers.partial_surrender` WHERE set_month = "202